# Extract all Odoo contacts

Connects to Odoo 18 via the `/json/2` JSON-RPC API and exports all `res.partner` contacts to a CSV file.

## Setup

Import dependencies, configure Odoo credentials, and define the `json2_call` helper for the `/json/2` API.

In [1]:
import json
import ssl
import urllib.request
import urllib.error
import pandas as pd
from pathlib import Path

ODOO_URL = "https://jupyter-lab.19.odoo.build/"
ODOO_DB = "jupyter-lab"
ODOO_USERNAME = "login@mint-system.ch"
ODOO_PASSWORD = ""

INPUT_DIR = Path("input")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

def json2_call(model, method, params, bearer_token=None):
    url = f"{ODOO_URL.rstrip('/')}/json/2/{model}/{method}"
    payload = json.dumps(params).encode()
    headers = {"Content-Type": "application/json"}
    if bearer_token:
        headers["Authorization"] = f"Bearer {bearer_token}"
    req = urllib.request.Request(url, data=payload, headers=headers)
    try:
        with urllib.request.urlopen(req, context=ssl_ctx) as resp:
            result = json.loads(resp.read())
            return result
    except urllib.error.HTTPError as e:
        raise Exception(f"HTTP {e.code}: {e.read().decode()}")

print("Setup complete.")

Setup complete.


## Test connection

Verify the API connection works by searching for contacts.

In [2]:
result = json2_call("res.partner", "search", {"domain": [], "offset": 0}, bearer_token=ODOO_PASSWORD)
print(f"Found {len(result)} contacts.")

Found 38 contacts.


## Export contacts

Fetch all contacts with fields `id`, `company_type`, `name`, `email`, `type`, `parent_id` and save to `input/res.partner.csv`.

In [3]:
fields = ["company_type", "name", "email", "type", "parent_id"]

partner_ids = json2_call("res.partner", "search", {"domain": [], "offset": 0}, bearer_token=ODOO_PASSWORD)
print(f"Found {len(partner_ids)} contacts.")

xml_id_map = json2_call("res.partner", "get_external_id", {"ids": partner_ids}, bearer_token=ODOO_PASSWORD)

contacts = []
BATCH_SIZE = 200
for i in range(0, len(partner_ids), BATCH_SIZE):
    batch = partner_ids[i:i + BATCH_SIZE]
    records = json2_call("res.partner", "read", {"ids": batch, "fields": fields}, bearer_token=ODOO_PASSWORD)
    for rec in records:
        db_id = rec["id"]
        rec["id"] = xml_id_map.get(db_id) or db_id
        if isinstance(rec.get("parent_id"), list) and len(rec["parent_id"]) == 2:
            rec["parent_id"] = rec["parent_id"][0]
    contacts.extend(records)

df = pd.DataFrame(contacts)
df.to_csv(INPUT_DIR / "res.partner.csv", index=False)
print(f"Exported {len(df)} contacts to input/res.partner.csv")

Found 38 contacts.


Exported 38 contacts to input/res.partner.csv
